# Lab 2: Country Schools Data Analysis

## 1. Introduction

This lab focuses on preprocessing the Country Schools dataset and applying anomaly detection and cluster analysis using Python. Anomaly detection identifies unusual school records, while cluster analysis groups schools with similar characteristics.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("Country Schools.csv")

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

Dataset loaded successfully.
Dataset shape: (50, 8)


In [2]:
# Inspect the dataset
print("First five rows:")
display(df.head())

print("Last five rows:")
display(df.tail())

print("\nColumn data types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

First five rows:


,SNO,TOTALSCHOOLS,TotalSchoolsInstalled,SchoolsPending,PercentageInstalled,InstalledLDDs,InstalledTDDs,FID
0,TURKANA,358.0,37.0,321.0,10.34%,3744.0,74.0,1
1,HOMA BAY,875.0,552.0,323.0,63.09%,22982.0,1104.0,2
2,WEST POKOT,498.0,242.0,256.0,48.59%,15114.0,484.0,3
3,KITUI,1238.0,1200.0,38.0,96.93%,42162.0,2400.0,4
4,TANA RIVER,159.0,143.0,16.0,89.94%,7415.0,286.0,5


Last five rows:


,SNO,TOTALSCHOOLS,TotalSchoolsInstalled,SchoolsPending,PercentageInstalled,InstalledLDDs,InstalledTDDs,FID
45,EMBU,381.0,248.0,133.0,65.09%,8438.0,496.0,46
46,SAMBURU,146.0,49.0,97.0,33.56%,2891.0,98.0,47
47,TOTAL,21729.0,14629.0,7100.0,67.32%,723618.0,29258.0,48
48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49
49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50



Column data types:
SNO                          str
TOTALSCHOOLS             float64
TotalSchoolsInstalled    float64
SchoolsPending           float64
PercentageInstalled          str
InstalledLDDs            float64
InstalledTDDs            float64
FID                        int64
dtype: object

Missing values per column:
SNO                      2
TOTALSCHOOLS             2
TotalSchoolsInstalled    2
SchoolsPending           2
PercentageInstalled      2
InstalledLDDs            2
InstalledTDDs            2
FID                      0
dtype: int64

Duplicate rows: 0


In [3]:
# Remove rows with missing values
df = df.dropna()

# Remove the overall TOTAL summary row
df = df[df["SNO"].str.strip().str.upper() != "TOTAL"].copy()

# Convert percentage text to numeric values
df["PercentageInstalled"] = (
    df["PercentageInstalled"]
    .str.replace("%", "", regex=False)
    .astype(float)
)

# Remove identifier columns that are not useful for analysis
df = df.drop(columns=["FID"])

# Reset row numbering
df = df.reset_index(drop=True)

# Check the cleaned dataset
print("Cleaned dataset shape:", df.shape)
print("\nMissing values:", df.isnull().sum().sum())
print("\nData types:")
print(df.dtypes)

display(df.head())

Cleaned dataset shape: (47, 7)

Missing values: 0

Data types:
SNO                          str
TOTALSCHOOLS             float64
TotalSchoolsInstalled    float64
SchoolsPending           float64
PercentageInstalled      float64
InstalledLDDs            float64
InstalledTDDs            float64
dtype: object


,SNO,TOTALSCHOOLS,TotalSchoolsInstalled,SchoolsPending,PercentageInstalled,InstalledLDDs,InstalledTDDs
0,TURKANA,358.0,37.0,321.0,10.34,3744.0,74.0
1,HOMA BAY,875.0,552.0,323.0,63.09,22982.0,1104.0
2,WEST POKOT,498.0,242.0,256.0,48.59,15114.0,484.0
3,KITUI,1238.0,1200.0,38.0,96.93,42162.0,2400.0
4,TANA RIVER,159.0,143.0,16.0,89.94,7415.0,286.0


In [4]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from sklearn.preprocessing import StandardScaler

# Select numerical features for analysis
features = df.drop(columns=["SNO"])

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

print("Number of counties:", X_scaled.shape[0])
print("Number of numerical features:", X_scaled.shape[1])
print("Data preparation completed.")

Number of counties: 47
Number of numerical features: 6
Data preparation completed.


In [6]:
from sklearn.ensemble import IsolationForest

# Create and train the anomaly detection model
model = IsolationForest(contamination=0.05, random_state=42)
df["Anomaly"] = model.fit_predict(X_scaled)

# Display the detected anomalies
anomalies = df[df["Anomaly"] == -1]
print("Number of anomalies detected:", len(anomalies))
display(anomalies[["SNO", "TOTALSCHOOLS",
                   "TotalSchoolsInstalled", "SchoolsPending",
                   "PercentageInstalled", "InstalledLDDs",
                   "InstalledTDDs"]])

Number of anomalies detected: 3


,SNO,TOTALSCHOOLS,TotalSchoolsInstalled,SchoolsPending,PercentageInstalled,InstalledLDDs,InstalledTDDs
0,TURKANA,358.0,37.0,321.0,10.34,3744.0,74.0
3,KITUI,1238.0,1200.0,38.0,96.93,42162.0,2400.0
17,KAKAMEGA,880.0,572.0,308.0,65.00,44338.0,1144.0


### Interpretation of Anomaly Detection

The Isolation Forest algorithm detected three unusual counties: Turkana, Kitui, and Kakamega. Turkana has a low school installation percentage of 10.34%, while Kitui has a high installation percentage of 96.93%. Kakamega has a relatively high number of installed LDDs (44,338). These records differ from other counties in some of their school-related characteristics. However, being identified as an anomaly does not necessarily mean that the data is incorrect.

In [7]:
from sklearn.cluster import KMeans

# Group counties into 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)

# Display counties and their assigned clusters
print("Number of counties in each cluster:")
print(df["Cluster"].value_counts().sort_index())

print("\nCounty assignments:")
display(df[["SNO", "Cluster"]].sort_values("Cluster"))

Number of counties in each cluster:
Cluster
0    16
1    17
2    14
Name: count, dtype: int64

County assignments:


,SNO,Cluster
1,HOMA BAY,0
3,KITUI,0
7,MIGORI,0
11,NANDI,0
13,MAKUENI,0
15,KISII,0
10,NAROK,0
16,KISUMU,0
19,BOMET,0
27,MERU,0


### Interpretation of Cluster Analysis

The K-Means algorithm grouped the 47 counties into three clusters. Cluster 0 contains 16 counties, Cluster 1 contains 17 counties, and Cluster 2 contains 14 counties.

The clusters group counties according to similarities in their numerical characteristics, including total schools, installed schools, pending schools, installation percentage, installed LDDs, and installed TDDs. Counties in the same cluster have more similar patterns across these features than counties in different clusters. Further comparison of cluster averages is needed to determine the main characteristics of each group.

In [8]:
# Compare average numerical values across clusters
cluster_summary = df.groupby("Cluster")[features.columns].mean().round(2)

display(cluster_summary)

,TOTALSCHOOLS,TotalSchoolsInstalled,SchoolsPending,PercentageInstalled,InstalledLDDs,InstalledTDDs
Cluster,,,,,,
0,742.62,542.56,200.06,72.69,27458.06,1085.12
1,309.94,244.65,65.29,80.36,12209.29,489.29
2,327.00,127.79,199.21,33.92,5480.79,255.57


### Interpretation of Cluster Characteristics

The cluster summary shows differences in school-related characteristics across the three groups.

- **Cluster 0 (16 counties):** Has the highest average total schools (742.62), installed schools (542.56), and installed LDDs (27,458.06). Its average installation percentage is 72.69%.
- **Cluster 1 (17 counties):** Has an average installation percentage of 80.36%, the highest among the three clusters. It also has the lowest average number of pending schools (65.29).
- **Cluster 2 (14 counties):** Has the lowest average installation percentage (33.92%) and the lowest average numbers of installed schools (127.79), installed LDDs (5,480.79), and installed TDDs (255.57). Its average number of pending schools is 199.21.

Overall, Cluster 0 contains counties with larger school counts and more installed devices, Cluster 1 shows the highest installation progress, and Cluster 2 shows lower installation progress. These differences demonstrate how clustering can help identify groups with different school infrastructure patterns.

## 9. Conclusion

The Country Schools dataset was preprocessed by removing missing records and the summary row, converting the installation percentage to a numerical format, and standardizing the numerical features.

Isolation Forest identified three unusual counties: Turkana, Kitui, and Kakamega. K-Means grouped the 47 counties into three clusters with different school counts, installation percentages, and device installations.

These experiments demonstrate how anomaly detection can identify unusual observations and how cluster analysis can reveal similarities and differences between counties. The results can support further investigation of school infrastructure and installation progress.